In [2]:
import geopandas as gpd
import pandas as pd

parquet_path = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_roads.parquet'

print("Loading Parquet file...")
gdf_roads = gpd.read_parquet(parquet_path)

print(f"\nOriginal shape: {gdf_roads.shape[0]} rows, {gdf_roads.shape[1]} columns")

# Calculate the minimum number of non-NaN values required to KEEP a column.
# If we drop >90% NaN, we want to KEEP columns that have at least 10% valid data.
min_valid_values = int(len(gdf_roads) * 0.10)

# Drop the columns (axis=1 means columns)
gdf_roads_clean = gdf_roads.dropna(axis=1, thresh=min_valid_values)

# Let's see what got removed
dropped_cols = set(gdf_roads.columns) - set(gdf_roads_clean.columns)

print(f"\nNew shape: {gdf_roads_clean.shape[0]} rows, {gdf_roads_clean.shape[1]} columns")
print(f"Removed {len(dropped_cols)} empty/sparse columns:")
for col in dropped_cols:
    print(f" - {col}")



Loading Parquet file...

Original shape: 656826 rows, 610 columns

New shape: 656826 rows, 14 columns
Removed 596 empty/sparse columns:
 - tiger:name_type_1
 - trolleybus
 - tiger:name_base_3
 - sidewalk:both:bicycle
 - placement:backward
 - side
 - indoor
 - tiger:zip_right_4
 - name:wam
 - parking:lane:left:parallel
 - destination:street:backward
 - source:start_date
 - parking:condition:right:maxstay:conditional
 - ski:nordic
 - hgv:forward
 - cycleway:left:oneway
 - expressway
 - lanes:bus:backward
 - parking:lane:right
 - width:lanes
 - tactile_paving
 - railway
 - dual_carriageway
 - addr:street
 - parking:both:maxstay
 - cycleway:left:width
 - traffic_control
 - service:bicycle:cleaning
 - bicycle
 - source:noname
 - old_name:1830
 - area:highway
 - loc_name
 - name_2
 - proposed
 - barrier
 - maxwidth
 - hand_cart
 - smoothness
 - name_en
 - sidewalk:right
 - construction_date
 - highway_1
 - sidewalk:both:surface
 - railway:name
 - name:source
 - turn:forward
 - cycleway:left


In [7]:
import geopandas as gpd
import pandas as pd

# (Assuming you already ran the code to create gdf_roads_clean)

# 1. Check how many rows we have before filtering
print(f"Rows before removing motorways: {len(gdf_roads_clean)}")

# 2. Filter and Merge in the 'highway' column
if 'highway' in gdf_roads_clean.columns:
    # A. Filter out 'motorway' and 'motorway_link'
    gdf_roads_clean = gdf_roads_clean[~gdf_roads_clean['highway'].isin(['motorway', 'motorway_link'])]
    print(f"Rows after removing motorways: {len(gdf_roads_clean)}")

    print("\nMerging highway classes...")

    # B. Handle "no class" (NaN or empty strings) -> 'service'
    gdf_roads_clean['highway'] = gdf_roads_clean['highway'].fillna('service')
    gdf_roads_clean['highway'] = gdf_roads_clean['highway'].replace('', 'service')

    # C. Map the classes
    # trunk + trunk_link -> trunk, and trunk -> primary.
    # Therefore: trunk, trunk_link, and primary_link all become 'primary'.
    # unclassified -> service
    mapping_dict = {
        'trunk': 'primary',
        'trunk_link': 'primary',
        'primary_link': 'primary',
        'unclassified': 'service'
    }

    # Apply the mapping
    gdf_roads_clean['highway'] = gdf_roads_clean['highway'].replace(mapping_dict)
    print("Merge complete! Top 10 highway classes are now:")
    print(gdf_roads_clean['highway'].value_counts().head(10))

else:
    print("Warning: Could not find a column named 'highway'. Check your column names!")

# 3. Define the new GeoPackage path
clean_gpkg_path = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_roads_clean.gpkg'

print(f"\nSaving cleaned dataset to GeoPackage...")
print("(Note: GeoPackages are SQLite databases, so this will take slightly longer to write than a Parquet file. Hang tight!)")

# 4. Save using the GPKG driver
gdf_roads_clean.to_file(clean_gpkg_path, driver="GPKG")

print(f"Saved successfully to: {clean_gpkg_path} ✅")

Rows before removing motorways: 646725
Rows after removing motorways: 646725

Merging highway classes...
Merge complete! Top 10 highway classes are now:
highway
service        433522
residential    156363
secondary       22043
primary         19398
tertiary        15399
Name: count, dtype: int64

Saving cleaned dataset to GeoPackage...
(Note: GeoPackages are SQLite databases, so this will take slightly longer to write than a Parquet file. Hang tight!)
Saved successfully to: /Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_roads_clean.gpkg ✅


In [7]:
import geopandas as gpd
import pandas as pd

def apply_exclusions():
    # 1. Paths
    roads_path = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_roads_clean.gpkg'
    exclusions_path = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/EXCLUSIONS/MA_exclusions.gpkg'

    # Saving to a new file so you don't accidentally overwrite your clean base data!
    output_path = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_roads_fully_excluded.gpkg'

    # 2. Load Data
    print("Loading roads and exclusions...")
    # Using pyogrio to speed up loading the GPKG
    roads = gpd.read_file(roads_path, engine="pyogrio")
    exclusions = gpd.read_file(exclusions_path, engine="pyogrio")

    # 3. Project to meters (MA State Plane) for accurate length math
    print("Projecting to EPSG:26986 (meters) for accurate length calculations...")
    roads = roads.to_crs("EPSG:26986")
    exclusions = exclusions.to_crs("EPSG:26986")

    # 4. Spatial Join to find interacting roads (The Bounding Box Filter)
    print("Using Spatial Index to find roads that touch the exclusion zones...")
    roads_intersecting = gpd.sjoin(roads, exclusions, how='inner', predicate='intersects')
    roads_to_cut_idx = roads_intersecting.index.unique()

    print(f"Total roads: {len(roads):,}")
    print(f"Roads interacting with exclusions: {len(roads_to_cut_idx):,}")

    # 5. Split dataset into Safe vs. Needs Cutting
    safe_roads = roads[~roads.index.isin(roads_to_cut_idx)].copy()
    roads_to_cut = roads[roads.index.isin(roads_to_cut_idx)].copy()

    # ==========================================
    # 6. Clean and Union Exclusions
    # ==========================================
    print("Cleaning invalid geometries in exclusions (fixing self-intersections)...")
    # Step A: Force valid geometries
    exclusions['geometry'] = exclusions.geometry.make_valid()

    # Step B: Apply a 0-meter buffer (The ultimate silver bullet for TopologyExceptions)
    exclusions['geometry'] = exclusions.geometry.buffer(0)

    # Step C: Drop any polygons that collapsed into empty shapes during cleaning
    exclusions = exclusions[~exclusions.geometry.is_empty]

    print("Unioning exclusion zones (this prevents overlapping exclusions from causing weird geometry bugs)...")
    if hasattr(exclusions.geometry, 'union_all'):
        exclusion_union = exclusions.geometry.union_all()
    else:
        exclusion_union = exclusions.unary_union

    # 7. Cut and Measure!
    print("Applying cuts and checking the 50% overlap rule...")
    roads_to_cut['original_length'] = roads_to_cut.geometry.length

    # Cookie-cutter the road
    roads_to_cut['geometry'] = roads_to_cut['geometry'].difference(exclusion_union)

    # Measure what is left
    roads_to_cut['new_length'] = roads_to_cut.geometry.length

    # Calculate overlap percentage
    roads_to_cut['overlap_pct'] = (roads_to_cut['original_length'] - roads_to_cut['new_length']) / roads_to_cut['original_length']

    # 8. Enforce the 50% Rule
    failed_mask = roads_to_cut['overlap_pct'] > 0.50
    failed_count = failed_mask.sum()
    print(f"Discarding {failed_count:,} roads because >50% of their length was inside exclusions.")

    # Keep roads with <= 50% overlap, and drop completely empty geometries
    roads_to_cut = roads_to_cut[~failed_mask & ~roads_to_cut['geometry'].is_empty].copy()

    # Clean up calculation columns
    roads_to_cut = roads_to_cut.drop(columns=['original_length', 'new_length', 'overlap_pct'])

    # 9. Recombine and Save
    print("Recombining the road network...")
    final_roads = pd.concat([safe_roads, roads_to_cut], ignore_index=True)

    print("Projecting back to EPSG:4326 (Standard GPS coordinates)...")
    final_roads = final_roads.to_crs("EPSG:4326")

    print(f"Saving final excluded roads to {output_path}...")
    final_roads.to_file(output_path, driver="GPKG")
    print("Done! ✅")

if __name__ == "__main__":
    apply_exclusions()

Loading roads and exclusions...
Projecting to EPSG:26986 (meters) for accurate length calculations...
Using Spatial Index to find roads that touch the exclusion zones...
Total roads: 646,725
Roads interacting with exclusions: 178,154
Cleaning invalid geometries in exclusions (fixing self-intersections)...
Unioning exclusion zones (this prevents overlapping exclusions from causing weird geometry bugs)...
Applying cuts and checking the 50% overlap rule...
Discarding 174,497 roads because >50% of their length was inside exclusions.
Recombining the road network...
Projecting back to EPSG:4326 (Standard GPS coordinates)...
Saving final excluded roads to /Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_roads_fully_excluded.gpkg...
Done! ✅


In [2]:
import geopandas as gpd
import pandas as pd
from collections import Counter

def process_road_network():
    # 1. Paths
    roads_path = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_roads_fully_excluded.gpkg'
    parkings_path = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_parkings.geojson'
    cemeteries_path = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/PUBLIC/MA_cemeteries.geojson'
    output_path = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_roads_final_excluded.gpkg'

    # 2. Load Roads and Project to Meters
    print("Loading roads...")
    roads = gpd.read_file(roads_path)

    print("Projecting to MA State Plane (meters) for accurate measurements...")
    roads = roads.to_crs("EPSG:26986")

    # ==========================================
    # STEP A: REMOVE SHORT DEAD-END STUMPS (TOPOLOGY FIX)
    # ==========================================
    print("\nAnalyzing network topology for dead-ends...")

    candidate_mask = roads['highway'].isin(['service', 'residential']) & (roads.geometry.length < 250)
    candidates = roads[candidate_mask].copy()
    print(f"Found {len(candidates):,} candidate short roads to check.")

    endpoints = candidates.geometry.boundary
    endpoints_gdf = gpd.GeoDataFrame(geometry=endpoints, crs=roads.crs).explode(index_parts=False)

    endpoints_gdf['endpoint_id'] = range(len(endpoints_gdf))
    endpoints_gdf['parent_road_id'] = endpoints_gdf.index
    endpoints_gdf['geometry'] = endpoints_gdf.geometry.buffer(0.1)

    print("Checking if endpoints physically touch other roads...")
    connections = gpd.sjoin(endpoints_gdf, roads, how='inner', predicate='intersects')

    valid_connections = connections[connections['parent_road_id'] != connections.index_right]
    connected_endpoint_ids = valid_connections['endpoint_id'].unique()
    endpoints_gdf['is_connected'] = endpoints_gdf['endpoint_id'].isin(connected_endpoint_ids)

    road_connectivity = endpoints_gdf.groupby('parent_road_id')['is_connected'].all()
    stump_ids = road_connectivity[road_connectivity == False].index

    roads = roads[~roads.index.isin(stump_ids)]
    print(f"Removed {len(stump_ids):,} short stump roads (kept T-junctions and through-streets)!")

    # ==========================================
    # STEP B: CUT ROADS INSIDE POLYGONS (TARGETED)
    # ==========================================
    print("\nLoading parking and cemetery polygons...")
    parkings = gpd.read_file(parkings_path, engine="pyogrio", use_arrow=True)
    cemeteries = gpd.read_file(cemeteries_path, engine="pyogrio", use_arrow=True)

    print("Aligning CRS and cleaning geometries...")
    parkings = parkings.to_crs(roads.crs)
    cemeteries = cemeteries.to_crs(roads.crs)

    # Combine exclusions
    exclusions = pd.concat([parkings[['geometry']], cemeteries[['geometry']]], ignore_index=True)

    # Wash geometries to prevent TopologyExceptions!
    exclusions['geometry'] = exclusions.geometry.make_valid()
    exclusions['geometry'] = exclusions.geometry.buffer(0)
    exclusions = exclusions[~exclusions.geometry.is_empty]

    print("Using Spatial Index to find the roads that actually touch exclusions...")
    roads_intersecting = gpd.sjoin(roads, exclusions, how='inner', predicate='intersects')
    interacting_idx = roads_intersecting.index.unique()

    # --- NEW LOGIC: Only target 'service' and 'residential' roads ---
    # Any other road type (like primary) will skip this entirely and remain "safe"
    target_mask = roads.index.isin(interacting_idx) & roads['highway'].isin(['service', 'residential'])

    print(f"Out of {len(roads):,} roads, {len(interacting_idx):,} interact with polygons.")
    print(f"However, only {target_mask.sum():,} are 'service'/'residential' and will be evaluated/cut.")

    # Split the dataset: safe roads (untouched) vs. roads we need to cut
    safe_roads = roads[~target_mask].copy()
    roads_to_cut = roads[target_mask].copy()

    print("Unioning exclusion zones for the final cut...")
    if hasattr(exclusions.geometry, 'union_all'):
        exclusion_union = exclusions.geometry.union_all()
    else:
        exclusion_union = exclusions.unary_union

    print("Erasing road segments and checking overlap percentage...")

    if len(roads_to_cut) > 0:
        roads_to_cut['original_length'] = roads_to_cut.geometry.length
        roads_to_cut['geometry'] = roads_to_cut['geometry'].difference(exclusion_union)
        roads_to_cut['new_length'] = roads_to_cut.geometry.length

        roads_to_cut['overlap_pct'] = (roads_to_cut['original_length'] - roads_to_cut['new_length']) / roads_to_cut['original_length']

        failed_roads_count = (roads_to_cut['overlap_pct'] > 0.50).sum()
        print(f"Discarding {failed_roads_count:,} roads because more than 50% of their length was inside an exclusion zone.")

        # Keep valid roads
        roads_to_cut = roads_to_cut[(roads_to_cut['overlap_pct'] <= 0.50) & (~roads_to_cut['geometry'].is_empty)]
        roads_to_cut = roads_to_cut.drop(columns=['original_length', 'new_length', 'overlap_pct'])

    print("Recombining the road network...")
    roads = pd.concat([safe_roads, roads_to_cut], ignore_index=True)

    # ==========================================
    # STEP C: SAVE
    # ==========================================
    print("\nProjecting back to standard GPS coordinates (EPSG:4326)...")
    roads = roads.to_crs("EPSG:4326")

    print(f"Saving final dataset to {output_path}...")
    roads.to_file(output_path, driver="GPKG")
    print("Done! ✅")

if __name__ == "__main__":
    process_road_network()


Loading roads...
Projecting to MA State Plane (meters) for accurate measurements...

Analyzing network topology for dead-ends...
Found 376,059 candidate short roads to check.
Checking if endpoints physically touch other roads...
Removed 216,977 short stump roads (kept T-junctions and through-streets)!

Loading parking and cemetery polygons...
Aligning CRS and cleaning geometries...
Using Spatial Index to find the roads that actually touch exclusions...
Out of 255,251 roads, 33,624 interact with polygons.
However, only 33,465 are 'service'/'residential' and will be evaluated/cut.
Unioning exclusion zones for the final cut...
Erasing road segments and checking overlap percentage...
Discarding 26,707 roads because more than 50% of their length was inside an exclusion zone.
Recombining the road network...

Projecting back to standard GPS coordinates (EPSG:4326)...
Saving final dataset to /Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_roads_final_excluded.gpkg...
Done!

In [3]:
import geopandas as gpd
import pandas as pd
import networkx as nx

def prune_road_network():
    # 1. Paths
    input_path = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_roads_final_excluded.gpkg'
    output_path = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_roads_pruned.gpkg'

    # 2. Load and Project
    print("Loading excluded road network...")
    roads = gpd.read_file(input_path, engine="pyogrio")
    roads = roads.to_crs("EPSG:26986")

    # ==========================================
    # STEP 1: EXPLODE MULTI-PART GEOMETRIES
    # ==========================================
    # This splits rows where multiple disconnected lines are grouped together into one row.
    # It exposes the "floating" segments so we can evaluate and delete them individually.
    print(f"\nExploding multi-part lines... (Original rows: {len(roads):,})")
    roads = roads.explode(index_parts=False).reset_index(drop=True)
    print(f"After explosion: {len(roads):,} discrete road segments.")

    # ==========================================
    # STEP 2: SECOND PASS STUMP REMOVAL
    # ==========================================
    print("\nRunning second pass on dead-end stumps and floating lines...")
    candidate_mask = roads['highway'].isin(['service', 'residential']) & (roads.geometry.length < 200)
    candidates = roads[candidate_mask].copy()

    # Get endpoints and buffer slightly for T-junctions
    endpoints = candidates.geometry.boundary
    endpoints_gdf = gpd.GeoDataFrame(geometry=endpoints, crs=roads.crs).explode(index_parts=False)
    endpoints_gdf['endpoint_id'] = range(len(endpoints_gdf))
    endpoints_gdf['parent_road_id'] = endpoints_gdf.index
    endpoints_gdf['geometry'] = endpoints_gdf.geometry.buffer(0.1)

    # Check connections
    connections = gpd.sjoin(endpoints_gdf, roads, how='inner', predicate='intersects')
    valid_connections = connections[connections['parent_road_id'] != connections.index_right]

    connected_endpoint_ids = valid_connections['endpoint_id'].unique()
    endpoints_gdf['is_connected'] = endpoints_gdf['endpoint_id'].isin(connected_endpoint_ids)

    # A line is a stump (or floating) if NOT ALL of its endpoints connect
    road_connectivity = endpoints_gdf.groupby('parent_road_id')['is_connected'].all()
    stump_ids = road_connectivity[road_connectivity == False].index

    roads = roads[~roads.index.isin(stump_ids)].reset_index(drop=True)
    print(f"Removed {len(stump_ids):,} stumps and floating lines!")

    # ==========================================
    # STEP 3: REMOVE ISOLATED ROAD CLUSTERS
    # ==========================================
    print("\nBuilding network graph to find isolated road clusters...")

    # We buffer all roads by 10cm to ensure we catch T-junctions as "touching"
    roads_buffered = gpd.GeoDataFrame(geometry=roads.geometry.buffer(0.1), crs=roads.crs)

    # Spatial join the network against itself! This creates a list of every road that touches another road.
    intersections = gpd.sjoin(roads_buffered, roads, how='inner', predicate='intersects')

    # Build a Graph where Nodes = Roads, and Edges = Intersections
    G = nx.Graph()
    # Add an edge for every pair of roads that intersect
    G.add_edges_from(zip(intersections.index, intersections.index_right))

    # Find all distinct connected subgraphs (clusters)
    clusters = list(nx.connected_components(G))
    print(f"Found {len(clusters):,} distinct disconnected road networks.")

    # Keep only clusters that have at least 100 connected road segments.
    # (Why not just keep the single largest? Because MA has islands like Martha's Vineyard!
    # If we only keep the absolute biggest network, we'd delete the islands entirely.)
    MIN_CLUSTER_SIZE = 100

    valid_road_indices = []
    dropped_clusters = 0

    for cluster in clusters:
        if len(cluster) >= MIN_CLUSTER_SIZE:
            valid_road_indices.extend(cluster)
        else:
            dropped_clusters += 1

    roads = roads.loc[valid_road_indices].reset_index(drop=True)
    print(f"Dropped {dropped_clusters:,} isolated clusters (small disconnected road webs).")
    print(f"Final network contains {len(roads):,} perfectly connected road segments.")

    # ==========================================
    # STEP 4: SAVE
    # ==========================================
    print("\nProjecting back to GPS coordinates (EPSG:4326)...")
    roads = roads.to_crs("EPSG:4326")

    print(f"Saving pristine network to {output_path}...")
    roads.to_file(output_path, driver="GPKG")
    print("Done! ✅")

if __name__ == "__main__":
    prune_road_network()

Loading excluded road network...

Exploding multi-part lines... (Original rows: 228,544)
After explosion: 231,693 discrete road segments.

Running second pass on dead-end stumps and floating lines...
Removed 28,693 stumps and floating lines!

Building network graph to find isolated road clusters...
Found 2,921 distinct disconnected road networks.
Dropped 2,907 isolated clusters (small disconnected road webs).
Final network contains 195,197 perfectly connected road segments.

Projecting back to GPS coordinates (EPSG:4326)...
Saving pristine network to /Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_roads_pruned.gpkg...
Done! ✅


In [7]:
import geopandas as gpd
import pandas as pd
import networkx as nx
from shapely.geometry import MultiLineString, Point
from shapely.ops import linemerge, substring
from collections import defaultdict

def merge_and_split_linestrings():
    # 1. Paths
    input_path = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_roads_pruned.gpkg'
    output_path = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_roads_stitched.gpkg'

    # 2. Load and Project
    print("Loading pruned roads...")
    roads = gpd.read_file(input_path, engine="pyogrio")
    print(f"Original total roads: {len(roads):,}")

    print("Projecting to MA State Plane (meters)...")
    roads = roads.to_crs("EPSG:26986")

    # ==========================================
    # STEP 1: MAP ALL ROAD CONNECTIONS
    # ==========================================
    print("\nMapping endpoints to find 2-degree connections...")
    pt_to_roads = defaultdict(list)

    for row in roads.itertuples():
        geom = row.geometry
        if geom is None or geom.is_empty or geom.geom_type != 'LineString':
            continue

        coords = list(geom.coords)
        pt_to_roads[coords[0]].append(row.Index)
        pt_to_roads[coords[-1]].append(row.Index)

    # ==========================================
    # STEP 2: BUILD THE MERGE GRAPH
    # ==========================================
    print("Identifying valid contiguous road segments...")
    G = nx.Graph()

    def names_match(n1, n2):
        if pd.isna(n1) and pd.isna(n2): return True
        return str(n1) == str(n2)

    for pt, connected_roads in pt_to_roads.items():
        if len(connected_roads) == 2:
            r1, r2 = connected_roads[0], connected_roads[1]
            if r1 == r2: continue

            hw1, hw2 = roads.at[r1, 'highway'], roads.at[r2, 'highway']
            name1 = roads.at[r1, 'name'] if 'name' in roads.columns else None
            name2 = roads.at[r2, 'name'] if 'name' in roads.columns else None

            if hw1 == hw2 and names_match(name1, name2):
                G.add_edge(r1, r2)

    # ==========================================
    # STEP 3: STITCH AND ENFORCE 3000m LIMIT
    # ==========================================
    print("\nStitching roads (max 3000m limit per stitch)...")
    roads_to_drop = set()
    new_roads_data = []

    components = list(nx.connected_components(G))

    for comp in components:
        subgraph = G.subgraph(comp)
        start_node = next((n for n, d in subgraph.degree() if d == 1), list(comp)[0])
        path_sequence = list(nx.dfs_preorder_nodes(subgraph, start_node))

        current_chunk = []
        current_len = 0.0

        for r_idx in path_sequence:
            l = roads.at[r_idx, 'geometry'].length

            if current_len + l > 3500 and len(current_chunk) > 0:
                if len(current_chunk) > 1:
                    roads_to_drop.update(current_chunk)
                    geoms = [roads.at[i, 'geometry'] for i in current_chunk]
                    merged_geom = linemerge(MultiLineString(geoms))

                    base_row = roads.loc[current_chunk[0]].copy()
                    base_row['geometry'] = merged_geom
                    new_roads_data.append(base_row)

                current_chunk = [r_idx]
                current_len = l
            else:
                current_chunk.append(r_idx)
                current_len += l

        if len(current_chunk) > 1:
            roads_to_drop.update(current_chunk)
            geoms = [roads.at[i, 'geometry'] for i in current_chunk]
            merged_geom = linemerge(MultiLineString(geoms))

            base_row = roads.loc[current_chunk[0]].copy()
            base_row['geometry'] = merged_geom
            new_roads_data.append(base_row)

    print(f"Sewed {len(roads_to_drop):,} fragments into {len(new_roads_data):,} pristine lines.")

    # Apply the stitched roads to a working dataset
    final_roads = roads.drop(index=list(roads_to_drop)).copy()
    if new_roads_data:
        new_roads_df = gpd.GeoDataFrame(new_roads_data, crs=roads.crs)
        final_roads = pd.concat([final_roads, new_roads_df], ignore_index=True)

    # ==========================================
    # STEP 4: SPLIT ROADS LONGER THAN 5KM
    # ==========================================
    print("\nScanning for massive roads (>5km) to split into <=3500m chunks...")
    final_roads['length'] = final_roads.geometry.length
    long_roads_mask = (final_roads['length'] > 5000) & (final_roads.geom_type == 'LineString')

    long_roads = final_roads[long_roads_mask].copy()
    safe_roads = final_roads[~long_roads_mask].copy()

    print(f"Found {len(long_roads)} roads exceeding 5km. Processing...")

    if len(long_roads) > 0:
        # Get all true junctions (endpoints of every road currently in the network)
        # Because we stitched degree-2 nodes, endpoints are now degree 1, 3, 4, etc.
        all_endpoints = set()
        for geom in final_roads.geometry:
            if geom is not None and not geom.is_empty and geom.geom_type == 'LineString':
                coords = list(geom.coords)
                all_endpoints.add(coords[0])
                all_endpoints.add(coords[-1])

        split_roads_data = []

        for idx, row in long_roads.iterrows():
            line = row.geometry
            total_len = line.length

            # Find which coordinates on this long line are actual network junctions
            line_coords = set(line.coords)
            junc_coords_on_line = line_coords.intersection(all_endpoints)

            # Calculate how far down the line each junction is
            junc_distances = sorted([line.project(Point(c)) for c in junc_coords_on_line])

            current_start = 0.0

            while total_len - current_start > 3500:
                target_dist = current_start + 3500

                # Find best junction between [current_start + 500m] and [target_dist]
                # We add 500m so we don't accidentally snap to a junction 2 meters away and make a tiny stub
                valid_juncs = [d for d in junc_distances if current_start + 500 < d <= target_dist]

                # If there's an intersection in that range, split there! Otherwise, force a cut at 3500m.
                if valid_juncs:
                    split_dist = max(valid_juncs)
                else:
                    split_dist = target_dist

                # Extract the fragment using Shapely's clean substring tool
                fragment = substring(line, current_start, split_dist)

                new_row = row.copy()
                new_row['geometry'] = fragment
                split_roads_data.append(new_row)

                current_start = split_dist

            # Save the final remaining tail piece of the road
            if current_start < total_len:
                fragment = substring(line, current_start, total_len)
                new_row = row.copy()
                new_row['geometry'] = fragment
                split_roads_data.append(new_row)

        print(f"Successfully split {len(long_roads)} massive roads into {len(split_roads_data)} smaller logical fragments.")

        # Combine back into the main dataset
        split_df = gpd.GeoDataFrame(split_roads_data, crs=final_roads.crs)
        final_roads = pd.concat([safe_roads, split_df], ignore_index=True)

    # Clean up the temp length column
    final_roads = final_roads.drop(columns=['length'])

    # ==========================================
    # STEP 5: SAVE
    # ==========================================
    print("\nProjecting back to Standard GPS (EPSG:4326)...")
    final_roads = final_roads.to_crs("EPSG:4326")

    print(f"\n---> FINAL TOTAL ROADS INSIDE DATASET: {len(final_roads):,} <---")

    print(f"Saving finalized dataset to {output_path}...")
    final_roads.to_file(output_path, driver="GPKG")
    print("Done! ✅")

if __name__ == "__main__":
    merge_and_split_linestrings()

Loading pruned roads...
Original total roads: 195,197
Projecting to MA State Plane (meters)...

Mapping endpoints to find 2-degree connections...
Identifying valid contiguous road segments...

Stitching roads (max 3000m limit per stitch)...
Sewed 54,955 fragments into 21,349 pristine lines.

Scanning for massive roads (>5km) to split into <=3500m chunks...
Found 175 roads exceeding 5km. Processing...
Successfully split 175 massive roads into 391 smaller logical fragments.

Projecting back to Standard GPS (EPSG:4326)...

---> FINAL TOTAL ROADS INSIDE DATASET: 161,807 <---
Saving finalized dataset to /Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_roads_stitched.gpkg...
Done! ✅


In [7]:
import geopandas as gpd
import pandas as pd
import networkx as nx
from shapely.geometry import Point, MultiLineString
from shapely.ops import linemerge, substring
from collections import defaultdict

def heal_network(edges_gdf):
    """Helper function to zip collinear segments sharing a degree-2 node."""
    print("Stitching collinear degree-2 segments back together...")
    pt_to_roads = defaultdict(list)

    for idx, row in edges_gdf.iterrows():
        coords = list(row.geometry.coords)
        pt_to_roads[coords[0]].append(idx)
        pt_to_roads[coords[-1]].append(idx)

    G = nx.Graph()
    G.add_nodes_from(edges_gdf.index)

    for pt, connected_roads in pt_to_roads.items():
        if len(connected_roads) == 2:
            r1, r2 = connected_roads[0], connected_roads[1]
            if r1 == r2: continue

            hw1, hw2 = edges_gdf.at[r1, 'highway'], edges_gdf.at[r2, 'highway']
            if hw1 == hw2:
                G.add_edge(r1, r2)

    components = list(nx.connected_components(G))
    final_edges_data = []

    for comp in components:
        comp_list = list(comp)
        if len(comp_list) == 1:
            final_edges_data.append(edges_gdf.loc[comp_list[0]])
        else:
            geoms = [edges_gdf.at[i, 'geometry'] for i in comp_list]
            merged_geom = linemerge(MultiLineString(geoms))

            if merged_geom.geom_type == 'MultiLineString':
                for sub_geom in merged_geom.geoms:
                    new_row = edges_gdf.loc[comp_list[0]].copy()
                    new_row['geometry'] = sub_geom
                    final_edges_data.append(new_row)
            else:
                new_row = edges_gdf.loc[comp_list[0]].copy()
                new_row['geometry'] = merged_geom
                final_edges_data.append(new_row)

    healed_gdf = gpd.GeoDataFrame(final_edges_data, crs=edges_gdf.crs).reset_index(drop=True)
    print(f"Healed! Graph contains {len(healed_gdf):,} logical edges.")
    return healed_gdf


def build_perfect_topology():
    # 1. Paths
    input_path = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_roads_stitched.gpkg'
    edges_output = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_network_edges.gpkg'
    nodes_output = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_network_nodes.gpkg'
    cleaned_stitched_output = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_roads_stitched_cleaned.gpkg'

    # 2. Load
    print("Loading stitched road network...")
    roads = gpd.read_file(input_path, engine="pyogrio")
    roads = roads.to_crs("EPSG:26986")
    roads['geometry'] = roads.geometry.make_valid()

    # We strictly explode here to guarantee there are NO MultiLineStrings going into our interval tracker
    roads = roads.explode(index_parts=False).reset_index(drop=True)
    roads = roads[roads.geometry.type == 'LineString'].copy()

    # ==========================================
    # STEP 1: PLANARIZE (Noding the Network)
    # ==========================================
    print("\nPlanarizing the network... (Fracturing lines exactly at intersections)")
    if hasattr(roads.geometry, 'union_all'):
        noded_geom = roads.geometry.union_all()
    else:
        noded_geom = roads.unary_union

    if hasattr(noded_geom, 'geoms'):
        edges_list = [geom for geom in noded_geom.geoms if geom.geom_type == 'LineString']
    elif noded_geom.geom_type == 'LineString':
        edges_list = [noded_geom]
    else:
        edges_list = []

    exploded_edges = gpd.GeoDataFrame(geometry=edges_list, crs=roads.crs)

    # ==========================================
    # STEP 2: RECOVER ATTRIBUTES
    # ==========================================
    print("Recovering road attributes via midpoints...")
    exploded_edges['midpoint'] = exploded_edges.geometry.interpolate(0.5, normalized=True)
    midpoints_gdf = gpd.GeoDataFrame(geometry=exploded_edges['midpoint'], crs=roads.crs)

    joined = gpd.sjoin_nearest(midpoints_gdf, roads[['highway', 'name', 'geometry']], how='left')
    joined = joined[~joined.index.duplicated(keep='first')]

    exploded_edges['highway'] = joined['highway']
    exploded_edges['name'] = joined['name']
    exploded_edges = exploded_edges.drop(columns=['midpoint'])

    # ==========================================
    # STEP 3: DEGREE-2 HEALING (Pass 1)
    # ==========================================
    print("\n--- PASS 1: HEALING ---")
    edges_gdf = heal_network(exploded_edges)

    # ==========================================
    # STEP 3.5: GRAPH CLEANING & SYNCING ORIGINAL ROADS
    # ==========================================
    print("\nScanning for stumps, self-loops, and parallel edges...")
    edges_gdf['length'] = edges_gdf.geometry.length

    # 1. FIND STUMPS
    endpoint_counts = defaultdict(int)
    for geom in edges_gdf.geometry:
        coords = list(geom.coords)
        endpoint_counts[coords[0]] += 1
        endpoint_counts[coords[-1]] += 1

    degree_1_pts = {pt for pt, count in endpoint_counts.items() if count == 1}
    stumps_to_drop = []

    for idx, row in edges_gdf.iterrows():
        if row['highway'] in ['service', 'residential'] and row['length'] < 300:
            coords = list(row.geometry.coords)
            if coords[0] in degree_1_pts or coords[-1] in degree_1_pts:
                stumps_to_drop.append(idx)

    # 2. FIND SELF-LOOPS & PARALLEL EDGES
    class_rank = {
        'motorway': 10, 'trunk': 9, 'primary': 8, 'secondary': 7,
        'tertiary': 6, 'unclassified': 5, 'residential': 4,
        'service': 3, 'track': 2, 'path': 1
    }

    edge_pairs = defaultdict(list)
    self_loops = []

    for idx, row in edges_gdf.iterrows():
        coords = list(row.geometry.coords)
        pt1, pt2 = coords[0], coords[-1]

        # Self-Loop Check
        if pt1 == pt2:
            self_loops.append(idx)
        else:
            # Parallel check
            pair_key = tuple(sorted((pt1, pt2)))
            rank = class_rank.get(str(row['highway']).lower(), 0)
            edge_pairs[pair_key].append((idx, rank))

    parallel_drops = []
    for pair_key, edges_list in edge_pairs.items():
        if len(edges_list) > 1:
            edges_list.sort(key=lambda x: x[1], reverse=True)
            for edge in edges_list[1:]:
                parallel_drops.append(edge[0])

    all_drops = list(set(stumps_to_drop + self_loops + parallel_drops))

    if all_drops:
        print(f"Flagged {len(stumps_to_drop):,} stumps, {len(self_loops):,} self-loops, and {len(parallel_drops):,} parallel edges.")
        print("Mapping drops back to original roads to mathematically slice them...")

        edges_to_drop_gdf = edges_gdf.loc[all_drops].copy()
        edges_to_drop_gdf['midpoint'] = edges_to_drop_gdf.geometry.interpolate(0.5, normalized=True)
        dropped_midpoints = gpd.GeoDataFrame(geometry=edges_to_drop_gdf['midpoint'], crs=roads.crs)

        joined = gpd.sjoin_nearest(dropped_midpoints, roads[['geometry']], how='left')
        joined = joined[~joined.index.duplicated(keep='first')]

        valid_intervals = {idx: [(0.0, geom.length)] for idx, geom in roads.geometry.items() if geom and geom.geom_type == 'LineString'}

        def remove_interval(intervals, drop_s, drop_e):
            new_intervals = []
            for (s, e) in intervals:
                if drop_e <= s + 0.01 or drop_s >= e - 0.01:
                    new_intervals.append((s, e))
                else:
                    if s < drop_s - 0.01:
                        new_intervals.append((s, drop_s))
                    if e > drop_e + 0.01:
                        new_intervals.append((drop_e, e))
            return new_intervals

        for idx in all_drops:
            if idx not in joined.index or pd.isna(joined.at[idx, 'index_right']):
                continue

            orig_idx = joined.at[idx, 'index_right']
            if orig_idx not in valid_intervals:
                continue

            orig_geom = roads.at[orig_idx, 'geometry']
            edge_geom = edges_to_drop_gdf.at[idx, 'geometry']

            # --- THE FIX: Using midpoint and length to handle self-loops safely ---
            mid_pt = edge_geom.interpolate(0.5, normalized=True)
            d_mid = orig_geom.project(mid_pt)
            half_len = edge_geom.length / 2.0

            drop_s = max(0.0, d_mid - half_len)
            drop_e = min(orig_geom.length, d_mid + half_len)

            valid_intervals[orig_idx] = remove_interval(valid_intervals[orig_idx], drop_s, drop_e)

        print("Reconstructing final synced geometries...")
        valid_roads_data = []
        for orig_idx, intervals in valid_intervals.items():
            orig_geom = roads.at[orig_idx, 'geometry']
            for s, e in intervals:
                if e - s > 0.05:
                    new_geom = substring(orig_geom, s, e)
                    new_row = roads.loc[orig_idx].copy()
                    new_row['geometry'] = new_geom
                    valid_roads_data.append(new_row)

        roads_cleaned = gpd.GeoDataFrame(valid_roads_data, crs=roads.crs).reset_index(drop=True)

        # Apply drops to the Edge Graph
        edges_gdf = edges_gdf.drop(index=all_drops).reset_index(drop=True)
        print(f"Sync complete! Original stitched geometries properly sliced. Cleaned count: {len(roads_cleaned):,}")
    else:
        print("No drops found. Graph is pristine.")
        roads_cleaned = roads.copy()

    edges_gdf = edges_gdf.drop(columns=['length'])

    # ==========================================
    # STEP 3.6: DEGREE-2 HEALING (Pass 2)
    # ==========================================
    # --- THE FIX: We run healing again to zip up the gaps left by the parallel drops! ---
    print("\n--- PASS 2: HEALING ---")
    edges_gdf = heal_network(edges_gdf)

    # ==========================================
    # STEP 4: EXTRACT STRICT NODES AND RELATIONSHIPS
    # ==========================================
    print("\nExtracting precise coordinate nodes and mapping relationships...")

    edges_gdf['edge_id'] = [f"E_{i:06d}" for i in range(len(edges_gdf))]

    coord_to_node_id = {}
    node_to_edges = defaultdict(list)
    node_to_neighbors = defaultdict(set)

    def get_exact_node_id(pt_coords):
        if pt_coords not in coord_to_node_id:
            coord_to_node_id[pt_coords] = f"N_{len(coord_to_node_id):06d}"
        return coord_to_node_id[pt_coords]

    edges_gdf['start_node'] = None
    edges_gdf['end_node'] = None

    for idx, row in edges_gdf.iterrows():
        coords = list(row.geometry.coords)

        u_id = get_exact_node_id(coords[0])
        v_id = get_exact_node_id(coords[-1])
        edge_id = row['edge_id']

        edges_gdf.at[idx, 'start_node'] = u_id
        edges_gdf.at[idx, 'end_node'] = v_id

        node_to_edges[u_id].append(edge_id)
        node_to_edges[v_id].append(edge_id)

        if u_id != v_id:
            node_to_neighbors[u_id].add(v_id)
            node_to_neighbors[v_id].add(u_id)

    # ==========================================
    # STEP 5: BUILD AND SAVE
    # ==========================================
    print(f"\nConstructing final Nodes dataset ({len(coord_to_node_id):,} true nodes)...")

    nodes_data = []
    for coord_key, n_id in coord_to_node_id.items():
        nodes_data.append({
            'node_id': n_id,
            'connected_edges': ", ".join(node_to_edges[n_id]),
            'connected_nodes': ", ".join(sorted(list(node_to_neighbors[n_id]))),
            'geometry': Point(coord_key[0], coord_key[1])
        })

    nodes_gdf = gpd.GeoDataFrame(nodes_data, crs=edges_gdf.crs)
    cols = ['edge_id', 'start_node', 'end_node', 'highway', 'name', 'geometry']
    edges_gdf = edges_gdf[cols]

    print("\nProjecting back to GPS coordinates (EPSG:4326) and saving...")
    edges_gdf = edges_gdf.to_crs("EPSG:4326")
    nodes_gdf = nodes_gdf.to_crs("EPSG:4326")
    roads_cleaned = roads_cleaned.to_crs("EPSG:4326")

    edges_gdf.to_file(edges_output, driver="GPKG")
    nodes_gdf.to_file(nodes_output, driver="GPKG")
    roads_cleaned.to_file(cleaned_stitched_output, driver="GPKG")

    print("\nPerfect network topology successfully built and synced! ✅")

if __name__ == "__main__":
    build_perfect_topology()

Loading stitched road network...

Planarizing the network... (Fracturing lines exactly at intersections)
Recovering road attributes via midpoints...

--- PASS 1: HEALING ---
Stitching collinear degree-2 segments back together...
Healed! Graph contains 342,596 logical edges.

Scanning for stumps, self-loops, and parallel edges...
Flagged 24,540 stumps, 6,887 self-loops, and 20,079 parallel edges.
Mapping drops back to original roads to mathematically slice them...
Reconstructing final synced geometries...
Sync complete! Original stitched geometries properly sliced. Cleaned count: 130,512

--- PASS 2: HEALING ---
Stitching collinear degree-2 segments back together...
Healed! Graph contains 239,303 logical edges.

Extracting precise coordinate nodes and mapping relationships...

Constructing final Nodes dataset (172,337 true nodes)...

Projecting back to GPS coordinates (EPSG:4326) and saving...

Perfect network topology successfully built and synced! ✅


In [ ]:
# add the extclusion lines,
# check exclusions
# remove polygons early on